# Drieu, Todorova & Zugaro (2018) on CRCNS hc-18: travel, place fields, theta and sleep

Paper. C. Drieu, R. Todorova, M. Zugaro (2018), "Nested sequences of hippocampal assemblies during behavior support subsequent sleep replay", Science 362:675-679, doi:10.1126/science.aat2952. The claims file (`claims/drieu-2018.yaml`) was written from the version of record and the author manuscript (HAL hal-02330673v1) for the Methods and the supplement.

Dataset. CRCNS hc-18 (doi:10.6080/K0Z899MM), "Bilateral recordings from dorsal hippocampal area CA1 from rats transported on a model train and sleeping". CRCNS data sets are not versioned: the version is recorded as `unversioned`, and the MD5 sums of the files read (from the data set's `checksums.md5`) are printed below so that a later change is detectable. The files are under `hc-18/live/` in the public bucket `crcnsarchive`; there is no other copy (no `old/` folder), so this is the only version that can be pinned, and it is the one the paper's citation (data descriptor version 0.61, 2018-11-27) describes. Each rat is one tar.gz of NeuroSuite files, streamed and reduced in memory by `drieu2018.py`; nothing is written to disk.

This notebook tests the claims on travel, place fields, theta and sleep sessions. The decoding claims are in `reproduce_drieu_2018_decoding.ipynb`.

| claim | tested here |
|---|---|
| figs1-travel-similar | yes |
| figs2b-place-cell-count | yes |
| fig1c-field-size-similar | yes |
| fig1c-peak-rate-lower-passive1 | yes |
| figs2c-track-coverage | yes |
| fig1d-no-remapping-p1-active | yes |
| fig1e-no-remapping-active-p2 | yes |
| figs2d-stability-p1-p2 | yes |
| fig2b-theta-frequency-unchanged | yes |
| fig2b-theta-power-reduced | yes |
| fig2b-theta-asymmetry-reduced | yes |
| fig2cd-spike-bursts-slower | yes |
| figs4b-proportions-oscillating-precessing | yes |
| fig2f-phase-precession-degraded | yes |
| figs4de-precession-range-similar | yes |
| figs11-sleep-sessions-similar | yes |
| all decoding claims (fig3*, figs5b, figs7 to figs10, fig4*, figs12) | in the decoding notebook |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time, resource, gc
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import drieu2018 as D

T_START = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
sums = D.checksums()
for n in ["code.zip"] + [f"data/{r}.tar.gz" for r in D.RATS]:
    print(f"{sums[n]}  {n}")
code = D.read_code_zip()
print(len(code), "MATLAB scripts in code.zip")

In [ ]:
RAT, PREP, LFPS = {}, {}, {}
for name in D.RATS:
    t = time.time()
    r = D.load_rat(name)
    P = D.prepare(r)
    LF = D.reduce_lfp(r, P)
    r["lfp_shape"] = r["lfp"]["data"].shape
    del r["lfp"]
    RAT[name], PREP[name], LFPS[name] = r, P, LF
    gc.collect()
    print(f"{name}: {time.time() - t:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

What was loaded, per rat (one session per rat), and the checks of the description against the values.

In [ ]:
rows = []
for name in D.RATS:
    r, P, LF = RAT[name], PREP[name], LFPS[name]
    m = r["meta"]
    rows.append(dict(rat=name[6:9], channels=m["nChannels"], spike_groups=len(m["groups"]), acq_Hz=m["fs"],
                     units=len(r["units"]), pyramidal=int(P["pyr"].sum()), isolated=int(P["iso"].sum()),
                     pyr_isolated=int((P["pyr"] & P["iso"]).sum()),
                     spikes_end_s=round(r["res_max_s"], 1), pos_end_s=round(len(r["pos"]) / D.POS_FS, 1),
                     lfp_end_s=round(r["lfp_shape"][1] / D.LFP_FS, 1),
                     pos_lost_pct=round(100 * np.mean(np.isnan(P["xy"][:, 0])), 1),
                     track_perimeter_cm=round(P["L"]), direction_flipped=P["flipped"],
                     theta_channel=LF["theta_channel"], pyr_layer_channels=len(LF["pyr_channels"])))
sanity = pd.DataFrame(rows)
print(sanity.to_string(index=False))
print()
print("Epochs (consolidated as GetEventsManip1.m does), minutes:")
rows = []
for name in D.RATS:
    ep = PREP[name]["ep"]
    d = {"rat": name[6:9]}
    for c in D.CONDS:
        a, b = ep["travel"][c]
        d[c] = round((b - a) / 60, 2)
    for s in D.SLEEPS:
        a, b = ep["sleep"][s]
        d[s] = round((b - a) / 60, 1)
    d["n_PF_blocks"] = len(ep["PF_blocks"]); d["n_PAF_blocks"] = len(ep["PAF_blocks"]); d["n_sleep_blocks"] = len(ep["SL_blocks"])
    rows.append(d)
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Event names per rat:")
for name in D.RATS:
    print(name[6:9], sorted({n.split("-")[-1] for n, a, b in RAT[name]["events"]}))
print()
print("Position ranges (cm) and velocity during travel:")
for name in D.RATS:
    P = PREP[name]
    xy = P["xy"]
    tr = D.in_intervals(P["t"], np.array([P["ep"]["travel"][c] for c in D.CONDS]))
    print(name[6:9], "x", np.round(np.nanpercentile(xy[tr, 0], [0, 100])), "y", np.round(np.nanpercentile(xy[tr, 1], [0, 100])),
          "velocity median/95th pct (cm/s)", np.round(np.nanpercentile(P["v"][tr], [50, 95]), 1),
          "moving periods", len(P["mov"]))

Clock checks. The spikes, positions and LFP of each rat end within a second of each other (table above). Below, place-field sharpness (spatial information) with spike times shifted against positions, and ripple rates in sleep and in SWS. The shift test turns out unable to detect an offset: the train moves at a near-constant speed around a closed track, so a time shift only rotates the fields, and the information barely changes (for example 0.245 bits at 0 s and 0.244 at -10 s in rat 242). That spikes, positions and LFP share a clock is therefore consistent with the data, not shown here. The decoding notebook gives further support: theta sequences there sweep ahead of the decoded current position, which needs spikes, LFP phase and positions aligned within a theta cycle.

In [ ]:
def spatial_info(P, c, shift):
    cd = P["cond"][c]
    ok = np.isfinite(P["lin_unw"])
    tt, lu = P["t"][ok], P["lin_unw"][ok]
    occ = np.bincount(np.minimum((cd["occ_lin"] * D.NBINS).astype(int), D.NBINS - 1), minlength=D.NBINS) / D.POS_FS
    po = occ / occ.sum()
    out = []
    for i in D.place_cells(P, c):
        st = PREP_UNITS[P["name"]][i]
        a, b = cd["interval"]
        st = st[(st >= a) & (st <= b)] + shift
        k = np.clip(np.round(st * D.POS_FS).astype(int), 0, len(P["t"]) - 1)
        st = st[P["mv"][k]]
        sl = np.mod(np.interp(st, tt, lu), 1)
        rate, _ = D.firing_curve(sl, cd["occ_lin"], 4.5 / (P["L"] / D.NBINS))
        lam = np.sum(po * rate)
        with np.errstate(divide="ignore", invalid="ignore"):
            out.append(np.nansum(po * rate / lam * np.log2(np.where(rate > 0, rate / lam, 1))))
    return np.mean(out)

PREP_UNITS = {name: [u["times"] for u in RAT[name]["units"]] for name in D.RATS}
shifts = [-10, -2, -0.5, 0, 0.5, 2, 10]
rows = []
for name in D.RATS:
    P = PREP[name]
    rows.append([name[6:9]] + [round(spatial_info(P, "A", s), 3) for s in shifts])
print("Spatial information (bits/spike), Active place cells, spike times shifted by (s):")
print(pd.DataFrame(rows, columns=["rat"] + [str(s) for s in shifts]).to_string(index=False))
rows = []
for name in D.RATS:
    P, LF = PREP[name], LFPS[name]
    rip_sleep = np.mean([len(LF["sleep"][s]["ripples"]) / np.diff(LF["sleep"][s]["interval"])[0] for s in D.SLEEPS])
    rip_sws = []
    for s in D.SLEEPS:
        rp, sw = LF["sleep"][s]["ripples"], LF["sleep"][s]["sws"]
        dur = np.sum(np.diff(sw, axis=1)) if len(sw) else np.nan
        rip_sws.append(np.sum(D.in_intervals(rp[:, 1], sw)) / dur if len(sw) else np.nan)
    rows.append(dict(rat=name[6:9], ripples_per_s_sleep=round(rip_sleep, 3), ripples_per_s_in_SWS=round(np.nanmean(rip_sws), 3)))
print(pd.DataFrame(rows).to_string(index=False))

## Units and place cells

Putative pyramidal cells: k-means on waveform width, trough-to-peak duration and firing rate (Methods, Data acquisition; claims file, Common methods). DiscriminateUnits.m clusters on the two durations only, and its rate exclusion of pyramidal cells above 5 Hz never applies: it tests `strcmp(rate,'on')` on the numeric array `rate` instead of `ratelim`, which is always false. The two classifications are compared below. Well-isolated: L-ratio < 0.05. Place cells per condition: well-isolated pyramidal cells with at least one field (Methods, Firing fields), under the field readings of the claims file: (a) no minimum field size, (b) 10 bins; and, as a variant, the settings of BatchPlaceFields.m (kernel 5 bins = 12 cm, minimum 10 bins, fields over 3/4 of the track dropped, at least 50 spikes in Active and 30 in Passive).

In [ ]:
rows = []
for name in D.RATS:
    r, P = RAT[name], PREP[name]
    pyr_code, _ = D.classify_units(r["units"], P["ep"]["travel"]["A"], reading="code")
    d = dict(rat=name[6:9], units=len(r["units"]), pyr_methods=int(P["pyr"].sum()), pyr_code=int(pyr_code.sum()),
             agree=int((P["pyr"] == pyr_code).sum()), pyr_iso=int((P["pyr"] & P["iso"]).sum()),
             median_waves_per_unit=int(np.median([u["n_wave"] for u in r["units"]])))
    for rd in ["a", "b", "code"]:
        for c in D.CONDS:
            d[f"{c}_{rd}"] = len(D.place_cells(P, c, rd))
    rows.append(d)
pc = pd.DataFrame(rows)
print(pc.to_string(index=False))
tot = pc.sum(numeric_only=True)
print()
for rd in ["a", "b", "code"]:
    print(f"reading {rd}: totals P1 {tot[f'P1_{rd}']}, A {tot[f'A_{rd}']}, P2 {tot[f'P2_{rd}']}  (paper 94, 150, 135; tolerance 10%)")
    long = pd.DataFrame([(r_, c, pc.loc[i, f"{c}_{rd}"]) for i, r_ in enumerate(pc.rat) for c in D.CONDS], columns=["rat", "cond", "n"])
    res = AnovaRM(long, "n", "rat", within=["cond"]).fit()
    print(res.anova_table.round(4).to_string())
    within = [abs(tot[f'{c}_{rd}'] - e) <= 0.1 * e for c, e in zip(D.CONDS, [94, 150, 135])]
    print("   within tolerance:", dict(zip(D.CONDS, within)), " P1 lowest:", tot[f'P1_{rd}'] < min(tot[f'A_{rd}'], tot[f'P2_{rd}']))

**Result: not reproduced.** The totals are 207, 203 and 228 place cells under field reading (a) and 183, 195 and 207 under (b), against 94, 150 and 135, so none is within 10%. The RM-ANOVA gives P = 0.055 and 0.128. Passive 1 is lowest under (b) only. The code's settings give 61, 104 and 100 (P = 0.092). The two unit classifications agree except in rat 292 (85 against 19 pyramidal cells).

## figs1-travel-similar

> Train velocity, number of laps and travel duration did not differ between Passive 1, Active and Passive 2 (repeated-measures ANOVA, P > 0.05 for each, 5 rats).

Velocity from tracked head position (LinearVelocity, s.d. 30 samples, as the scripts); readings (a) all position samples of the travel, (b) samples above 5 cm/s. Laps: completed turns of the unwrapped linear position. Duration: the travel session's interval (the events file), and, as a variant, the time spent moving.

In [ ]:
rows = []
for name in D.RATS:
    P = PREP[name]
    for c in D.CONDS:
        a, b = P["ep"]["travel"][c]
        s = (P["t"] >= a) & (P["t"] <= b)
        v = P["v"][s]
        lu = P["lin_unw"][s]
        lu = lu[np.isfinite(lu)]
        rows.append(dict(rat=name[6:9], cond=c, vel_a=np.nanmean(v), vel_b=np.nanmean(v[v > D.MINV]),
                         laps=np.floor(abs(lu[-1] - lu[0])), dur_min=(b - a) / 60,
                         moving_min=np.sum(D.in_intervals(P["t"][s], P["mov"])) / D.POS_FS / 60))
tv = pd.DataFrame(rows)
print(tv.round(2).to_string(index=False))
expected = {"vel": [46.80, 47.87, 46.73], "laps": [38.2, 30.8, 32.2], "dur_min": [8.24, 6.50, 7.20]}
tol = {"vel": [2.04, 1.83, 1.85], "laps": [2.8, 2.24, 1.77], "dur_min": [0.69, 0.33, 0.51]}
print()
for meas, col in [("vel", "vel_a"), ("vel", "vel_b"), ("laps", "laps"), ("dur_min", "dur_min"), ("dur_min", "moving_min")]:
    mean = tv.groupby("cond")[col].mean().reindex(D.CONDS)
    sem = tv.groupby("cond")[col].sem().reindex(D.CONDS)
    p = AnovaRM(tv, col, "rat", within=["cond"]).fit().anova_table["Pr > F"].iloc[0]
    f = AnovaRM(tv, col, "rat", within=["cond"]).fit().anova_table["F Value"].iloc[0]
    ok = [abs(m - e) <= t for m, e, t in zip(mean, expected[meas], tol[meas])]
    print(f"{col:11s} means {np.round(mean.values, 2)} (s.e.m. {np.round(sem.values, 2)}) expected {expected[meas]}; within tolerance {ok}; RM-ANOVA F = {f:.2f}, P = {p:.3f}")

**Result: not reproduced.** Velocity above 5 cm/s (45.6, 47.3 and 45.4 cm/s, P = 0.487) and laps (37.4, 29.4 and 31.2, P = 0.053) match, with every mean within tolerance. Travel duration differs across conditions under both readings (events file P = 0.023, time moving P = 0.045), so the claim of absence fails for duration. Over all samples, velocity is 42.2 and 43.4 cm/s in the passive conditions, outside tolerance.

## Firing fields: size, peak rate, coverage

> fig1c-field-size-similar: Place field size was the same in the three travel conditions (Kruskal-Wallis, P > 0.05).

> fig1c-peak-rate-lower-passive1: Place-cell peak firing rates were lower in Passive 1 than in Active.

> figs2c-track-coverage: In every travel condition the place fields together covered the whole track.

In [ ]:
def field_table(rd):
    rows = []
    for name in D.RATS:
        P = PREP[name]
        bin_cm = 600 / D.NBINS
        for c in D.CONDS:
            for i in D.place_cells(P, c, rd):
                f = P["cond"][c]["cells"][i]["readings"][rd]
                rate = f["rate"]
                for k, m in enumerate(f["fields"]):
                    rows.append(dict(rat=name[6:9], cond=c, cell=i, field=k, size_cm=m.sum() * bin_cm, peak=rate.max()))
    return pd.DataFrame(rows)

FT = {rd: field_table(rd) for rd in ["a", "b", "code"]}
for rd in ["a", "b", "code"]:
    ft = FT[rd]
    for sub, lab in [(ft[ft.field == 0], "peak field only"), (ft, "every field")]:
        g = [sub[sub.cond == c].size_cm.values for c in D.CONDS]
        h, p = stats.kruskal(*g)
        print(f"reading {rd:4s} {lab:15s}: median size (cm) {[round(np.median(x), 1) for x in g]}, n {[len(x) for x in g]}; KW H = {h:.2f}, P = {p:.4f}")
print("Field size per rat, median cm, reading a, peak field:")
print(FT["a"][FT["a"].field == 0].groupby(["rat", "cond"]).size_cm.median().unstack().reindex(columns=D.CONDS).round(1).to_string())

**Result: reproduced, resting on an assumption.** With the code's 10-bin minimum field size and the field holding the peak, Kruskal-Wallis gives P = 0.070 (medians 139, 170 and 134 cm). Without a minimum size P = 0.0065, and counting every field P = 0.0017 or less, so the outcome depends on the field-size reading.

In [ ]:
for rd in ["a", "b", "code"]:
    ft = FT[rd][FT[rd].field == 0]
    g = {c: ft[ft.cond == c].peak.values for c in D.CONDS}
    h, p = stats.kruskal(*g.values())
    u, p_pa = stats.mannwhitneyu(g["P1"], g["A"], alternative="two-sided")
    u2, p_ap2 = stats.mannwhitneyu(g["A"], g["P2"], alternative="two-sided")
    print(f"reading {rd:4s}: median peak rate (Hz) P1 {np.median(g['P1']):.2f}, A {np.median(g['A']):.2f}, P2 {np.median(g['P2']):.2f}; "
          f"KW P = {p:.2e}; rank-sum P1 vs A P = {p_pa:.2e} (Bonferroni x3: {min(1, 3 * p_pa):.2e}); A vs P2 P = {p_ap2:.2e}")
print("Per rat, median peak rate, reading a:")
print(FT["a"][FT["a"].field == 0].groupby(["rat", "cond"]).peak.median().unstack().reindex(columns=D.CONDS).round(2).to_string())

**Result: reproduced.** Median peak rates are 2.81, 3.95 and 3.00 Hz (reading b). The post hoc rank-sum test of Passive 1 against Active gives Bonferroni P = 1.3 x 10^-4, below 0.001/3, and reading (a) gives P = 1.7 x 10^-6. Passive 1 is lower than Active in every rat.

In [ ]:
for rd in ["a", "b", "code"]:
    rows = []
    for c in D.CONDS:
        pooled = np.zeros(D.NBINS, bool)
        per = []
        for name in D.RATS:
            P = PREP[name]
            cov = np.zeros(D.NBINS, bool)
            for i in D.place_cells(P, c, rd):
                for m in P["cond"][c]["cells"][i]["readings"][rd]["fields"]:
                    cov |= m
            per.append(100 * cov.mean())
            pooled |= cov
        rows.append(dict(cond=c, pooled_pct=round(100 * pooled.mean(), 1), **{f"rat_{n[6:9]}": round(x, 1) for n, x in zip(D.RATS, per)}))
    print(f"reading {rd}:")
    print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Fields cover 100% of the track in every condition, pooled and in each rat, under readings (a) and (b).

## Remapping

> fig1d-no-remapping-p1-active: Place fields did not remap between Passive 1 and Active; the mean unsigned proportional field shift was smaller than under random remapping (P < 0.01, 2000 bootstraps), and peak-location shifts were non-uniform and centred on zero (Rayleigh P < 0.001, V-test P < 10^-4).

> fig1e-no-remapping-active-p2: Place fields did not remap between Active and Passive 2 (bootstrap P ~ 0; Rayleigh P < 10^-9; V-test P < 10^-10).

> figs2d-stability-p1-p2: Place fields were also stable between Passive 1 and Passive 2 (bootstrap P < 0.02; V-test P < 0.05), though the angular shifts did not depart from uniformity by the Rayleigh test (P > 0.05).

Following the criterion: shift = lag of the peak of the cross-correlogram of the two firing curves over the mean field size; the mean unsigned shift against 2000 shuffles of cell identity in the test condition. Cells pooled over rats. Readings: cells with a field in both conditions or in either; circular or linear correlogram. Field readings a and b.

In [ ]:
def remap(c1, c2, rd, cellset, circular, rng):
    curves1, curves2, s1, s2, ang = [], [], [], [], []
    for name in D.RATS:
        P = PREP[name]
        pcs1, pcs2 = set(D.place_cells(P, c1, rd)), set(D.place_cells(P, c2, rd))
        cells = (pcs1 & pcs2) if cellset == "both" else (pcs1 | pcs2)
        for i in sorted(cells):
            f1 = P["cond"][c1]["cells"][i]["readings"][rd]
            f2 = P["cond"][c2]["cells"][i]["readings"][rd]
            a1 = f1["fields"][0].sum() if f1["fields"] else 0
            a2 = f2["fields"][0].sum() if f2["fields"] else 0
            if a1 == 0:
                a1 = a2
            if a2 == 0:
                a2 = a1
            curves1.append(f1["rate"]); curves2.append(f2["rate"]); s1.append(a1); s2.append(a2)
            d = (np.argmax(f2["rate"]) - np.argmax(f1["rate"])) / D.NBINS * 2 * np.pi
            ang.append(np.angle(np.exp(1j * d)))
    k = len(curves1)
    C1 = np.array(curves1); C2 = np.array(curves2)
    # lag matrix for every pair (i, j): curve1 of cell i against curve2 of cell j
    n = D.NBINS
    A = C1 - C1.mean(1, keepdims=True); B = C2 - C2.mean(1, keepdims=True)
    lag = np.zeros((k, k))
    if circular:
        FA, FB = np.conj(np.fft.fft(A, axis=1)), np.fft.fft(B, axis=1)
        lags = np.r_[np.arange(0, n // 2 + 1), np.arange(-(n - n // 2 - 1), 0)]
        for i in range(k):
            cc = np.real(np.fft.ifft(FA[i][None, :] * FB, axis=1))
            lag[i] = lags[np.argmax(cc, 1)]
    else:
        lags = np.arange(-(n - 1), n)
        for i in range(k):
            for j in range(k):
                lag[i, j] = lags[np.argmax(np.correlate(B[j], A[i], mode="full"))]
    s1, s2 = np.array(s1, float), np.array(s2, float)
    msize = (s1[:, None] + s2[None, :]) / 2
    prop = np.abs(lag) / msize
    obs = np.mean(np.diag(prop))
    null = np.array([np.mean(prop[np.arange(k), rng.permutation(k)]) for _ in range(2000)])
    p = np.mean(null <= obs)
    ang = np.array(ang)
    R, pr = D.rayleigh(ang)
    u, pv = D.vtest(ang)
    return dict(n=k, obs=obs, null_mean=null.mean(), null_min=null.min(), p_boot=p, rayleigh_R=R, rayleigh_p=pr, v_u=u, v_p=pv,
                mean_angle_deg=np.degrees(np.angle(np.mean(np.exp(1j * ang)))))

REMAP = {}
for pair, seed in [(("P1", "A"), 11), (("A", "P2"), 12), (("P1", "P2"), 13)]:
    rows = []
    for rd in ["a", "b"]:
        for cellset in ["both", "either"]:
            for circ in [True, False]:
                rng = np.random.default_rng(seed)
                res = remap(pair[0], pair[1], rd, cellset, circ, rng)
                rows.append(dict(fields=rd, cells=cellset, correlogram="circular" if circ else "linear", **res))
    REMAP[pair] = pd.DataFrame(rows)

def show_remap(pair):
    df = REMAP[pair].copy()
    for col in ["rayleigh_p", "v_p"]:
        df[col] = df[col].map(lambda x: f"{x:.1e}")
    print(f"control {pair[0]}, test {pair[1]}")
    print(df.round(3).to_string(index=False))

In [ ]:
show_remap(("P1", "A"))

**Result: partially reproduced, resting on an assumption.** The Rayleigh (P = 1.2 x 10^-4) and V-tests (P = 1.1 x 10^-5) reach the legend's bounds with fields of 10 bins or more and cells with fields in both conditions. The bootstrap reaches 0.05 only with the linear correlogram (P = 0.014), not 0.01. Under the other readings it gives P = 0.088 to 0.996.

In [ ]:
show_remap(("A", "P2"))

**Result: partially reproduced, resting on an assumption.** With fields of 10 bins or more, cells in both conditions and the linear correlogram, the bootstrap gives P = 0 (none of 2000), Rayleigh P = 1.6 x 10^-6 and V-test P = 1.5 x 10^-7. All three are below 0.001 but above the legend's 10^-9 and 10^-10.

In [ ]:
show_remap(("P1", "P2"))

**Result: partially reproduced, resting on an assumption.** The bootstrap (P = 0.016) and the V-test (P = 0.0016) hold under the same reading. The Rayleigh test finds non-uniformity under every reading (P = 0.0039 to 0.011), where the paper finds none.

## Theta frequency, power and asymmetry

> fig2b-theta-frequency-unchanged: Theta frequency did not differ between travel conditions (repeated-measures ANOVA, P > 0.90).

> fig2b-theta-power-reduced: Theta power was lower during passive travel than active travel (repeated-measures ANOVA, P < 10^-6), with power reduced at both the fundamental and the first harmonic.

> fig2b-theta-asymmetry-reduced: The shape of theta cycles changed during passive travel, with decreased cycle asymmetry (Kruskal-Wallis, P < 10^-84).

Multitaper spectra of the detrended LFP in 2-s windows (Methods, Theta power, frequency and cycles). LFP readings: (a) the pyramidal-layer channel with the largest theta power, (b) the mean of the pyramidal-layer channels. Samples: (all) the whole travel session, (mov) moving periods only.

In [ ]:
SPEC = {}
for name in D.RATS:
    P, LF = PREP[name], LFPS[name]
    for c in D.CONDS:
        tr = LF["travel"][c]
        n = len(tr["a"])
        tt = tr["t0"] + np.arange(n) / D.LFP_FS
        mvm = D.in_intervals(tt, P["mov"])
        for ch in ["a", "b"]:
            x = tr[ch]
            for samp in ["all", "mov"]:
                if samp == "all":
                    segs = [x]
                else:
                    i0, i1 = D.runs(mvm)
                    segs = [x[i:j] for i, j in zip(i0, i1) if j - i > 2 * D.LFP_FS]
                f, S, Sw = D.mt_psd(segs)
                SPEC[(name, c, ch, samp)] = (f, S, Sw)
f = SPEC[(D.RATS[0], "P1", "a", "all")][0]
th_band = (f >= 6) & (f <= 10)
harm = (f >= 12) & (f <= 20)

def theta_table(ch, samp):
    rows = []
    for name in D.RATS:
        S = np.array([SPEC[(name, c, ch, samp)][1] for c in D.CONDS])
        mu, sd = S.mean(0), S.std(0, ddof=1)
        Sn = (S - mu) / sd
        for k, c in enumerate(D.CONDS):
            rows.append(dict(rat=name[6:9], cond=c, freq=f[th_band][np.argmax(S[k, th_band])],
                             power=Sn[k, th_band].max(), harmonic=Sn[k, harm].max(), raw_power=S[k, th_band].max()))
    return pd.DataFrame(rows)

In [ ]:
for ch in ["a", "b"]:
    for samp in ["all", "mov"]:
        tt_ = theta_table(ch, samp)
        res = AnovaRM(tt_, "freq", "rat", within=["cond"]).fit().anova_table
        print(f"LFP {ch}, samples {samp}: theta frequency (Hz) by condition {tt_.groupby('cond').freq.mean().reindex(D.CONDS).round(2).to_dict()}; "
              f"RM-ANOVA F = {res['F Value'].iloc[0]:.2f}, P = {res['Pr > F'].iloc[0]:.3f}")
print(theta_table("a", "mov").pivot(index="rat", columns="cond", values="freq").reindex(columns=D.CONDS).to_string())
print("Frequency resolution of the 2-s spectra:", f[1] - f[0], "Hz")

**Result: reproduced.** The RM-ANOVA gives P = 0.13 to 0.41 across the LFP and sample readings (frequencies 6.8 to 7.1 Hz). With 0.5 Hz resolution most rats have the same mode in every condition, which makes this a weak test.

In [ ]:
def posthoc_paired(df, col):
    out = {}
    w = df.pivot(index="rat", columns="cond", values=col)
    for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
        t, p = stats.ttest_rel(w[a], w[b])
        out[f"{a}-{b}"] = (round(t, 2), min(1, 3 * p))
    return out

for ch in ["a", "b"]:
    for samp in ["all", "mov"]:
        tt_ = theta_table(ch, samp)
        for col in ["power", "harmonic"]:
            res = AnovaRM(tt_, col, "rat", within=["cond"]).fit().anova_table
            print(f"LFP {ch}, {samp}, {col:8s}: means {tt_.groupby('cond')[col].mean().reindex(D.CONDS).round(2).to_dict()}; "
                  f"RM-ANOVA P = {res['Pr > F'].iloc[0]:.2e}; post hoc paired t (t, Bonferroni P) {posthoc_paired(tt_, col)}")
# reading (b) of the unit: one value per 2-s window, normalized with the rat's condition mean and s.d.
for ch in ["a"]:
    for samp in ["mov"]:
        rows = []
        for name in D.RATS:
            S = np.array([SPEC[(name, c, ch, samp)][1] for c in D.CONDS])
            mu, sd = S.mean(0), S.std(0, ddof=1)
            for c in D.CONDS:
                Sw = SPEC[(name, c, ch, samp)][2]
                Wn = (Sw - mu) / sd
                for v in Wn[:, th_band].max(1):
                    rows.append(dict(rat=name[6:9], cond=c, power=v))
        wdf = pd.DataFrame(rows)
        fit = smf.ols("power ~ C(cond) + C(rat)", data=wdf).fit()
        at = sm.stats.anova_lm(fit, typ=2)
        print(f"per-window reading, LFP {ch}, {samp}: n windows {wdf.groupby('cond').size().reindex(D.CONDS).to_dict()}; "
              f"medians {wdf.groupby('cond').power.median().reindex(D.CONDS).round(2).to_dict()}; two-way ANOVA (condition + rat) condition P = {at.loc['C(cond)', 'PR(>F)']:.2e}")
        for a, b in [("P1", "A"), ("A", "P2")]:
            print(f"   rank-sum {a} vs {b}: P = {stats.mannwhitneyu(wdf[wdf.cond == a].power, wdf[wdf.cond == b].power).pvalue:.2e}")
print(theta_table("a", "mov").pivot(index="rat", columns="cond", values="raw_power").reindex(columns=D.CONDS).round(0).to_string())

**Result: partially reproduced, resting on an assumption.** Active has the highest normalized theta power under every reading. The ANOVA reaches 0.05 only with the mean of the pyramidal-layer channels (P = 0.036 moving, 0.052 all samples) or with one value per window (P = 0.0017). The largest-theta channel gives P = 0.18 and 0.29, and no paired post hoc test reaches 0.05/3. The 12-20 Hz power does not differ (P = 0.49 to 0.77).

In [ ]:
ASYM = {}
for ch in ["a", "b"]:
    rows = []
    for name in D.RATS:
        P, LF = PREP[name], LFPS[name]
        for c in D.CONDS:
            tr = LF["travel"][c]
            x = tr[ch]
            st, en, th = D.theta_cycles(x)
            tt = tr["t0"] + st / D.LFP_FS
            keep = D.in_intervals(tt, P["mov"])
            y, pk, trg = D.extrema_1_40(x)
            asy = D.asymmetry(st[keep], en[keep], y, pk, trg)
            for v in asy[np.isfinite(asy)]:
                rows.append((name[6:9], c, v))
    ASYM[ch] = pd.DataFrame(rows, columns=["rat", "cond", "asym"])
for ch in ["a", "b"]:
    df = ASYM[ch]
    g = {c: df[df.cond == c].asym.values for c in D.CONDS}
    h, p = stats.kruskal(*g.values())
    print(f"LFP {ch}: cycles {[len(v) for v in g.values()]}; median asymmetry {[round(np.median(v), 3) for v in g.values()]}; KW H = {h:.1f}, P = {p:.1e}")
    for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
        print(f"   rank-sum {a} vs {b}: P (Bonferroni x3) = {min(1, 3 * stats.mannwhitneyu(g[a], g[b]).pvalue):.1e}")
print(ASYM["a"].groupby(["rat", "cond"]).asym.median().unstack().reindex(columns=D.CONDS).round(3).to_string())

**Result: reproduced.** Median asymmetry is -0.104, -0.187 and -0.094, closer to zero in both passive conditions. Kruskal-Wallis gives P = 3.8 x 10^-22, and both Active post hoc comparisons give Bonferroni P < 10^-16. LFP reading (b) gives P = 2.2 x 10^-78 in the same direction. Rat 272 does not show the effect.

## Theta bursts and phase precession

> fig2cd-spike-bursts-slower: Place cells oscillated faster than theta during active travel, while during passive travel their spike bursts recurred at lower frequencies, closer to theta frequency.

> figs4b-proportions-oscillating-precessing: Fewer place cells oscillated faster than theta in passive travel (36/94 Passive 1, 90/150 Active, 63/135 Passive 2), and fewer were significantly phase precessing (14/94, 53/150, 21/135).

> fig2f-phase-precession-degraded: The distribution of phase precession slopes over all place cells differed between passive and active travel, indicating an overall degradation of phase precession in passive travel.

> figs4de-precession-range-similar: Among significantly phase-precessing cells, phase precession range and slopes did not differ between passive and active travel (Kruskal-Wallis and KS tests, P > 0.05).

Per place cell (field reading a) and condition, spikes during moving periods; theta phase from the theta channel (LFP reading a) under the two phase readings of the claims file. Burst frequency: spectrum of the spike train in units of theta cycles. Precession: in-field spikes, position normalized to the field (0 at entry, 1 at exit in the running direction), linear-circular regression with slopes searched within +/-2 pi rad per field (a standard range: one full cycle of precession over a field and more); at least 10 in-field spikes (a standard minimum).

In [ ]:
PH = []
for name in D.RATS:
    P, LF = PREP[name], LFPS[name]
    ok = np.isfinite(P["lin_unw"])
    for c in D.CONDS:
        tr = LF["travel"][c]
        x = tr["a"]
        tt = tr["t0"] + np.arange(len(x)) / D.LFP_FS
        uph = {rd: D.unwrapped_phase(x, rd) for rd in ["a", "b"]}
        for i in D.place_cells(P, c, "a"):
            cell = P["cond"][c]["cells"][i]
            st = cell["spk_t"]
            st = st[(st >= tt[0]) & (st <= tt[-1])]
            fld = cell["readings"]["a"]["fields"][0]
            # field entry in the running direction: first bin of the circular run
            idx = np.flatnonzero(fld)
            if fld.all():
                start, size = 0, D.NBINS
            else:
                sh = np.flatnonzero(~fld)[-1] + 1
                start, size = sh % D.NBINS, fld.sum()
            b = np.minimum((cell["spk_lin"][(cell["spk_t"] >= tt[0]) & (cell["spk_t"] <= tt[-1])] * D.NBINS), D.NBINS - 1e-9)
            rel = np.mod(b - start, D.NBINS) / size
            inf = rel < 1
            d = dict(rat=name[6:9], cond=c, cell=i, n_spikes=len(st), n_infield=int(inf.sum()))
            for rd in ["a", "b"]:
                u = np.interp(st, tt, uph[rd])
                d[f"burst_{rd}"] = D.spike_train_spectrum_peak(u / (2 * np.pi))
                if inf.sum() >= 10:
                    a_, phi0, rho, p = D.circ_lin_regression(rel[inf], np.mod(u[inf], 2 * np.pi))
                    d[f"slope_{rd}"], d[f"rho_{rd}"], d[f"p_{rd}"] = a_, rho, p
                else:
                    d[f"slope_{rd}"] = d[f"rho_{rd}"] = d[f"p_{rd}"] = np.nan
            PH.append(d)
PH = pd.DataFrame(PH)
print("place cells per condition:", PH.groupby("cond").size().reindex(D.CONDS).to_dict(),
      "; with >= 10 in-field spikes:", PH[PH.n_infield >= 10].groupby("cond").size().reindex(D.CONDS).to_dict())

In [ ]:
for rd in ["a", "b"]:
    col = f"burst_{rd}"
    g = {c: PH[(PH.cond == c)][col].dropna().values for c in D.CONDS}
    h, p = stats.kruskal(*g.values())
    print(f"phase reading {rd}: cells with a mode in 0.75-1.25 {[len(v) for v in g.values()]} of {PH.groupby('cond').size().reindex(D.CONDS).tolist()}; "
          f"median mode (cycle^-1) {[round(np.median(v), 4) for v in g.values()]}; KW P = {p:.1e}")
    for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
        pp = stats.mannwhitneyu(g[a], g[b]).pvalue
        print(f"   rank-sum {a} vs {b}: P = {pp:.1e} (Bonferroni x3 {min(1, 3 * pp):.1e})")
print(PH.groupby(["rat", "cond"]).burst_a.median().unstack().reindex(columns=D.CONDS).round(4).to_string())

**Result: not reproduced.** The spectral modes sit at 1.000 cycle^-1 in every condition (medians 0.9999, 1.0003 and 1.0000, Kruskal-Wallis P = 0.62). Under phase reading (b) Active is above Passive 1 (P = 0.0061) but not above Passive 2 (P = 0.41).

In [ ]:
def prop_test(k1, n1, k2, n2):
    p = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))

for rd in ["a", "b"]:
    n = PH.groupby("cond").size().reindex(D.CONDS)
    osc = PH.assign(o=PH[f"burst_{rd}"] > 1).groupby("cond").o.sum().reindex(D.CONDS)
    prec = PH.assign(o=(PH[f"p_{rd}"] < 0.05) & (PH[f"slope_{rd}"] < 0)).groupby("cond").o.sum().reindex(D.CONDS)
    prec_any = PH.assign(o=(PH[f"p_{rd}"] < 0.05)).groupby("cond").o.sum().reindex(D.CONDS)
    print(f"phase reading {rd}:")
    print("  oscillating faster than theta:", {c: f"{osc[c]}/{n[c]} = {osc[c] / n[c]:.3f}" for c in D.CONDS}, " expected 0.383, 0.600, 0.467")
    print("  significant negative precession:", {c: f"{prec[c]}/{n[c]} = {prec[c] / n[c]:.3f}" for c in D.CONDS}, " expected 0.149, 0.353, 0.156")
    print("  significant regression, either sign:", {c: f"{prec_any[c]}/{n[c]} = {prec_any[c] / n[c]:.3f}" for c in D.CONDS})
    for lab, k in [("osc", osc), ("prec", prec)]:
        for a in ["P1", "P2"]:
            z, p = prop_test(k["A"], n["A"], k[a], n[a])
            print(f"   binomial proportion test {lab}, A vs {a}: z = {z:.2f}, P = {p:.2e}")

**Result: not reproduced.** The proportions oscillating faster than theta are 0.488, 0.581 and 0.500. The Active excess is not significant against Passive 1 (P = 0.058) or Passive 2 (P = 0.091) under reading (a), and not against Passive 2 under (b) (P = 0.20). Significant precession is lower than the paper's (0.053, 0.192 and 0.105) but highest in Active (P = 1.7 x 10^-5 and 0.011).

In [ ]:
for rd in ["a", "b"]:
    col = f"slope_{rd}"
    g = {c: PH[PH.cond == c][col].dropna().values for c in D.CONDS}
    print(f"phase reading {rd}: n {[len(v) for v in g.values()]}; median slope (rad per field) {[round(np.median(v), 3) for v in g.values()]}")
    for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
        ks = stats.ks_2samp(g[a], g[b])
        print(f"   KS {a} vs {b}: D = {ks.statistic:.3f}, P = {ks.pvalue:.2e}")

**Result: partially reproduced, resting on an assumption.** Under phase reading (a) the Active median slope is the most negative (-0.82 against -0.37 and -0.30 rad per field). It differs from Passive 2 (KS P = 0.0055) but not from Passive 1 (P = 0.066). Reading (b) gives no difference (P = 0.12 and 0.63).

In [ ]:
for rd in ["a", "b"]:
    sig = PH[(PH[f"p_{rd}"] < 0.05) & (PH[f"slope_{rd}"] < 0)]
    g = {c: sig[sig.cond == c][f"slope_{rd}"].values for c in D.CONDS}
    rng_ = {c: np.degrees(np.abs(v)) for c, v in g.items()}
    print(f"phase reading {rd}: significant cells {[len(v) for v in g.values()]}")
    if min(len(v) for v in g.values()) >= 3:
        print(f"   range (deg per field) medians {[round(np.median(v), 1) for v in rng_.values()]}; KW P = {stats.kruskal(*rng_.values()).pvalue:.3f}")
        print(f"   slopes medians {[round(np.median(v), 3) for v in g.values()]}; KW P = {stats.kruskal(*g.values()).pvalue:.3f}")
        for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
            ks = stats.ks_2samp(g[a], g[b])
            print(f"   KS slopes {a} vs {b}: D = {ks.statistic:.3f}, P = {ks.pvalue:.3f}")

**Result: not reproduced, resting on an assumption.** Under phase reading (a) one test finds a difference, the KS test of Active against Passive 2 slopes (P = 0.043). The range and slope Kruskal-Wallis tests give P = 0.116. Under reading (b) every test gives P > 0.05.

## figs11-sleep-sessions-similar

> SWS duration, number and rate of candidate events and ripple rate did not differ between sleep sessions (repeated-measures ANOVA, P > 0.05), and candidate events coincided with ripples alike in all sessions (KS, P > 0.05).

SWS from the theta channel; ripples from the pyramidal-layer channels; candidate events from the place cells (field reading a) of the travel session that precedes each post-travel sleep. Readings of the event threshold: mean and s.d. over (sws) SWS bins, as BatchCandidateEvents.m does, or (all) the whole sleep session. Ripple threshold 3 s.d. for every rat (Methods); BatchFindRipples.m uses 4 for rat 272.

In [ ]:
SLEEP_OF = {"S1": "P1", "SA": "A", "S2": "P2"}
EVENTS = {}
rows = []
for name in D.RATS:
    P, LF = PREP[name], LFPS[name]
    for s in D.SLEEPS:
        sl = LF["sleep"][s]
        sws_min = np.sum(np.diff(sl["sws"], axis=1)) / 60 if len(sl["sws"]) else 0.0
        d = dict(rat=name[6:9], sleep=s, sws_min=sws_min, session_min=np.diff(sl["interval"])[0] / 60)
        rip = sl["ripples"]
        rip_sws = rip[D.in_intervals(rip[:, 1], sl["sws"])] if len(sl["sws"]) else rip[:0]
        d["ripples_per_min_sws"] = len(rip_sws) / sws_min if sws_min > 0 else np.nan
        if s in SLEEP_OF:
            c = SLEEP_OF[s]
            spk = np.sort(np.concatenate([PREP_UNITS[name][i] for i in D.place_cells(P, c, "a")]))
            for norm in ["sws", "all"]:
                ev = D.candidate_events(spk, sl["interval"], sl["sws"], norm=norm)
                EVENTS[(name, s, norm)] = ev
                d[f"events_{norm}"] = len(ev)
                d[f"events_per_min_{norm}"] = len(ev) / sws_min if sws_min > 0 else np.nan
                if len(ev) and len(rip_sws):
                    j = np.searchsorted(rip_sws[:, 1], ev[:, 1])
                    lo = rip_sws[np.clip(j - 1, 0, len(rip_sws) - 1), 1]
                    hi = rip_sws[np.clip(j, 0, len(rip_sws) - 1), 1]
                    dt = np.where(np.abs(ev[:, 1] - lo) < np.abs(ev[:, 1] - hi), ev[:, 1] - lo, ev[:, 1] - hi)
                    EVENTS[(name, s, norm, "dt")] = dt
                    d[f"frac_within_0.1s_{norm}"] = np.mean(np.abs(dt) < 0.1)
        rows.append(d)
SL = pd.DataFrame(rows)
print(SL.round(3).to_string(index=False))
print()
res = AnovaRM(SL, "sws_min", "rat", within=["sleep"]).fit().anova_table
print(f"SWS duration, 4 sessions: means {SL.groupby('sleep').sws_min.mean().reindex(D.SLEEPS).round(1).to_dict()}; RM-ANOVA P = {res['Pr > F'].iloc[0]:.3f}")
post = SL[SL.sleep != "SB"]
for col in ["events_sws", "events_per_min_sws", "events_all", "events_per_min_all", "ripples_per_min_sws"]:
    if post[col].isna().any():
        print(f"{col}: undefined for some rat (no SWS)"); continue
    res = AnovaRM(post, col, "rat", within=["sleep"]).fit().anova_table
    print(f"{col:22s}: means {post.groupby('sleep')[col].mean().reindex(['S1', 'SA', 'S2']).round(2).to_dict()}; RM-ANOVA P = {res['Pr > F'].iloc[0]:.3f}")
for norm in ["sws", "all"]:
    dts = {s: np.concatenate([EVENTS.get((n, s, norm, "dt"), np.zeros(0)) for n in D.RATS]) for s in SLEEP_OF}
    w = {s: v[np.abs(v) <= 0.5] for s, v in dts.items()}
    print(f"event peak minus nearest ripple peak (within +/-0.5 s), threshold over {norm}: n {[len(v) for v in w.values()]}, median (ms) {[round(1000 * np.median(v), 1) for v in w.values()]}")
    for a, b in [("S1", "SA"), ("SA", "S2"), ("S1", "S2")]:
        ks = stats.ks_2samp(w[a], w[b])
        print(f"   KS {a} vs {b}: D = {ks.statistic:.3f}, P = {ks.pvalue:.3f}")

**Result: not reproduced.** SWS duration (P = 0.96), candidate event counts (P = 0.69) and rates (P = 0.63), and ripple rates (P = 0.57) do not differ. The KS tests of event-to-ripple timing find a difference under both readings: P = 0.011 (after Passive 1 against after Active) and P = 0.025 (after Passive 1 against after Passive 2), with small effects (D = 0.028).

## Summary

| claim | outcome |
|---|---|
| figs1-travel-similar | not reproduced (duration differs) |
| figs2b-place-cell-count | not reproduced |
| fig1c-field-size-similar | reproduced (assumed 10-bin minimum field) |
| fig1c-peak-rate-lower-passive1 | reproduced |
| figs2c-track-coverage | reproduced |
| fig1d-no-remapping-p1-active | partially reproduced (assumed reading) |
| fig1e-no-remapping-active-p2 | partially reproduced (assumed reading) |
| figs2d-stability-p1-p2 | partially reproduced (assumed reading) |
| fig2b-theta-frequency-unchanged | reproduced |
| fig2b-theta-power-reduced | partially reproduced (assumed LFP reading) |
| fig2b-theta-asymmetry-reduced | reproduced |
| fig2cd-spike-bursts-slower | not reproduced |
| figs4b-proportions-oscillating-precessing | not reproduced |
| fig2f-phase-precession-degraded | partially reproduced (assumed phase reading) |
| figs4de-precession-range-similar | not reproduced (assumed phase reading) |
| figs11-sleep-sessions-similar | not reproduced (event-ripple timing differs) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method. The LFP channels, the linearization and the place-cell lists are not in the data set and were rebuilt (see the record). Place cells here are two to three times more numerous in Passive 1 than the paper's. Several results depend on choices the Methods leave open and the code settles differently: the field kernel and size limits, and the unit classification.

About the dataset. Spikes, positions and LFP end within a second of each other in every rat. Positions are lost for 40.6% and 22.8% of samples in rats 292 and 314. The traced track perimeter is 656 to 706 cm against the paper's 6 m. Rats 292 and 314 were recorded at 20 kHz. The scripts in code.zip load channel lists, theta cycles and place-cell lists that are not included.


In [ ]:
print(f"notebook runtime {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")